# German Credit · Riesgo de impago

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolenki7/German_Credit_Scoring_Data/blob/main/German_Credit_Data.ipynb)

## La pregunta de negocio

Un banco recibe **1.000 solicitudes de crédito** y sabe que **3 de cada 10 no se devuelven**. Las dos pérdidas no son simétricas: un impago cuesta el principal prestado, mientras que rechazar a un buen cliente solo cuesta el margen del préstamo. La pregunta no es *cuánto* riesgo hay, sino **qué variables lo separan y dónde se concentra**.

Este cuaderno responde:

1. **Dónde** se concentra el impago (cuenta corriente, plazo, edad).
2. **Qué** variables separan un buen crédito de un impago, en odds ratios legibles para negocio.
3. **Cuánto** acierta el modelo, con especial atención al recall de los impagos: un falso negativo es el error caro.

**Datos.** [Statlog German Credit (UCI)](https://archive.ics.uci.edu/ml/datasets/statlog+(german+credit+data)): 1.000 solicitudes y 20 variables. La versión ya limpia está versionada en el repo (`german_credit_cleaned.csv`), así que este cuaderno no descarga nada.


In [ ]:
import pathlib
import sqlite3

import numpy as np
import pandas as pd

# El generador de SVG vive en src/. En Colab el repo no está clonado, así que
# se descarga el módulo en vez de pedirle al lector que lo copie a mano.
try:
    from src.svg_chart import AZUL, ROJO, barras
except ImportError:
    import urllib.request
    url = ('https://raw.githubusercontent.com/Nicolenki7/'
           'German_Credit_Scoring_Data/main/src/svg_chart.py')
    exec(urllib.request.urlopen(url).read().decode())

df = pd.read_csv('german_credit_cleaned.csv')
print(f'{df.shape[0]} filas x {df.shape[1]} columnas')
print('target:', df['target'].value_counts().to_dict())
print('valores nulos:', int(df.isna().sum().sum()))


## 1. Preparación

Una sola decisión que merece explicarse: **la variable objetivo llega como texto** (`good` / `bad`). Se convierte a 1 = buen crédito y 0 = impago, y a partir de aquí el modelo estima **P(buen crédito)**. Esto importa para leer los coeficientes: un coeficiente **negativo** significa **más riesgo** de impago, no menos.

El resto llega ya normalizado del repositorio, así que aquí solo se comprueba que no haya nulos antes de seguir.


In [ ]:
df['target'] = (df['target'].astype(str).str.lower() == 'good').astype(int)
malo = 1 - df['target'].mean()
print(f'Tasa de impago: {malo:.1%} ({int((df["target"] == 0).sum())} de {len(df)})')
print(f'Préstamo mediano: {df["loan_amt"].median():,.0f} DM · plazo medio: {df["duration"].mean():.1f} meses')


## 2. Feature engineering en SQL

Los datos pasan por una base SQLite en memoria para construir tres columnas con `CASE WHEN`: el **ratio deuda/plazo** (detecta préstamos agresivos), el **tramo de edad** y el **tramo de plazo**. Son los dos últimos los que después explican casi todo el riesgo, y tenerlos como columnas permite graficarlos sin repetir la lógica.


In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('credit_data', conn, index=False, if_exists='replace')

sql = '''
SELECT *,
  CAST(loan_amt AS REAL) / duration AS Debt_to_Duration_Ratio,
  CASE WHEN age < 25 THEN 'Joven (<25)'
       WHEN age < 45 THEN 'Adulto (25-44)'
       ELSE 'Senior (45+)' END AS Tramo_Edad,
  CASE WHEN duration <= 12 THEN 'Corto (<=12m)'
       WHEN duration <= 36 THEN 'Medio (13-36m)'
       ELSE 'Largo (>36m)' END AS Tramo_Plazo
FROM credit_data;
'''

d = pd.read_sql_query(sql, conn)
conn.close()
print(f'{d.shape[1]} columnas tras el SQL')


## 3. Dónde se concentra el riesgo

Antes de modelar, la forma del riesgo en dos cortes. Los gráficos se guardan como SVG en `docs/img/`, así que lo que se ve aquí es exactamente lo que ilustra el README.


In [ ]:
IMG = pathlib.Path('docs/img')

ETIQUETA_CUENTA = {'below_0': 'sin fondo / <0 DM', 'below_200': '0-200 DM',
                  'above:200': '>200 DM', 'no_cheking_acc': 'sin cuenta'}
riesgo_cuenta = 1 - d.groupby('checking_acc_status')['target'].mean()
orden = ['below_0', 'below_200', 'above:200', 'no_cheking_acc']
riesgo_cuenta = riesgo_cuenta.reindex([c for c in orden if c in riesgo_cuenta.index])

barras(
    [(ETIQUETA_CUENTA[c], v, ROJO if v > malo else AZUL) for c, v in riesgo_cuenta.items()],
    'Sin cuenta o en números rojos: el doble de riesgo',
    IMG / 'credit-cuenta.svg',
    subtitulo=f'Tasa de impago por cuenta corriente · media {malo:.1%}',
    formato='{:.1%}')

riesgo_plazo = 1 - d.groupby('Tramo_Plazo')['target'].mean()
riesgo_plazo = riesgo_plazo.reindex(['Corto (<=12m)', 'Medio (13-36m)', 'Largo (>36m)'])
barras(
    [(c, v, ROJO if v > malo else AZUL) for c, v in riesgo_plazo.items()],
    'Los plazos largos concentran el impago',
    IMG / 'credit-plazo.svg',
    subtitulo=f'Tasa de impago por duración del préstamo · media {malo:.1%}',
    formato='{:.1%}')

print((riesgo_cuenta.reindex(orden)).round(3).to_string())
print()
print(riesgo_plazo.round(3).to_string())


## 4. El modelo

Regresión logística con `class_weight='balanced'` para que el 30 % de impagos no quede diluido, y **estandarización de las numéricas**: sin escalar, el solver no converge y los signos de los coeficientes dejan de tener sentido (la cuenta en rojo aparecía como factor protector). Las dummies ya viven en 0/1 y no se tocan.

Se reporta **AUC** y el **recall de la clase impago**, porque en riesgo crediticio el error caro es dejar pasar un mal crédito.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = d.drop(columns=['target', 'Tramo_Edad', 'Tramo_Plazo'])
y = d['target']
X = pd.get_dummies(X, columns=X.select_dtypes(exclude='number').columns.tolist(),
                  drop_first=True)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
numericas = X.select_dtypes(include='number').columns.tolist()
modelo = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), numericas)],
                              remainder='passthrough')),
    ('lr', LogisticRegression(max_iter=5000, class_weight='balanced')),
])
modelo.fit(X_tr, y_tr)
prob = modelo.predict_proba(X_te)[:, 1]
print(f'AUC {roc_auc_score(y_te, prob):.3f} · accuracy {accuracy_score(y_te, prob > .5):.3f}')
print(f'train {len(X_tr)} / test {len(X_te)}')
print()
print(classification_report(y_te, modelo.predict(X_te),
                            target_names=['impago', 'bueno'], digits=2))


In [ ]:
coefs = pd.DataFrame({'feature': X.columns, 'coef': modelo.named_steps['lr'].coef_[0]})
coefs['odds_ratio'] = np.exp(coefs['coef'])
coefs = coefs.reindex(coefs['coef'].abs().sort_values(ascending=False).index)
coefs.to_csv('outputs/credit_feature_importance.csv', index=False)

top = coefs.head(8)[::-1]
barras(
    [(f.replace('_', ' ')[:34], c, ROJO if c < 0 else AZUL)
     for f, c in zip(top['feature'], top['coef'])],
    'Qué separa un buen crédito de un impago',
    IMG / 'credit-drivers.svg',
    subtitulo='coeficiente logístico sobre P(buen crédito) · rojo = más riesgo',
    formato='{:+.2f}')

top.iloc[::-1][['feature', 'coef', 'odds_ratio']].to_string(index=False)


## Qué dicen los números

- **30.0 % de impago** (300 de 1.000). El préstamo mediano es de 2.320 DM a 20,9 meses de plazo medio.
- **La cuenta corriente es el mejor predictor simple:** 49,3 % de impago con la cuenta en números rojos frente a 11,7 % sin cuenta, pasando por 39,0 % (0-200 DM) y 22,2 % (>200 DM).
- **El plazo casi triplica el riesgo:** 21,2 % en préstamos de un año o menos frente a **51,7 %** en los de más de 36 meses.
- **La edad importa menos de lo que se supone:** 40,9 % en menores de 25 frente a 24,9 % en mayores de 45. El salto grande está entre jóvenes y el resto, no entre adultos y mayores.
- **AUC 0,772** con un modelo lineal. El dato que de verdad importa para negocio es el **recall de impagos: 0,68**, con una precisión de 0,50 — de cada 100 impagos reales detecta 68, y de cada 100 alertas 50 acaban siendo impago.
- **Los peores propósitos** son educación (44,0 %), 'otros' (41,7 %), coche nuevo (38,0 %) y reparaciones (36,4 %).

### Limitaciones

- **El dataset contiene una variable sensible** (`is_foreign_worker`) con un coeficiente alto. Describe este conjunto de datos de los años 90, **no** una política de crédito: usarla para decidir sería discriminación, y en la UE sería ilegal.
- Un solo *split* 75/25 con `random_state=42` y 250 casos de prueba: el AUC y el recall tienen mucha variación no medida. Con 1.000 filas, un intervalo de confianza amplio es inevitable.
- Los coeficientes son asociaciones, no causas, y las numéricas están estandarizadas: su magnitud es *por desviación típica*, comparable entre sí pero no en unidades de euros.
- El modelo predice P(buen crédito). Si se usa para decidir, el umbral debería fijarse con el coste real de cada error, no en 0,5.

### Siguiente paso

Sustituir el umbral fijo de 0,5 por una matriz de costes: cuánto cuesta un impago frente a cuánto margen se pierde al rechazar un buen cliente. Con eso la salida deja de ser una probabilidad y pasa a ser una decisión con un precio.
